# RoadSafe AI — Step 1: Complete Data Audit
### Proactive Road Risk Intelligence Platform

**Objective**: Perform a rigorous, reproducible Data Engineering audit of the **US Accidents (2016–2023)** dataset prior to any feature engineering or ML model training.

**Key Scientific Principles**:
1. Strict leakage prevention (identify and exclude all post-event features).
2. Verify exposure availability (traffic volume is absent and must not be fabricated).
3. Investigate the negative-sampling dilemma for incident-only datasets.

In [1]:
import os
import json
import pandas as pd
import numpy as np

DATA_PATH = os.path.join("..", "data", "US_Accidents_March23.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("data", "US_Accidents_March23.csv")

print(f"Target dataset path: {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")
if os.path.exists(DATA_PATH):
    print(f"File size on disk: {os.path.getsize(DATA_PATH) / (1024**3):.2f} GB")

## 1. Schema & Sample Inspection
Inspect the first few rows to observe the available columns and raw structure.

In [2]:
# Load preview sample of 5 rows
sample_df = pd.read_csv(DATA_PATH, nrows=5)
print(f"Total columns: {len(sample_df.columns)}")
print(sample_df.columns.tolist())
sample_df.head()

## 2. Load Audited Aggregate Statistics
Because the full dataset contains **7,728,394 rows** (~3.06 GB), we load the verified full-table streaming audit generated by `src/data_audit.py`.

In [3]:
report_path = os.path.join("..", "reports", "data_audit_report.json")
if not os.path.exists(report_path):
    report_path = os.path.join("reports", "data_audit_report.json")

with open(report_path, "r") as f:
    audit = json.load(f)

print(f"Total Verified Records: {audit['total_rows']:,}")
print(f"Total Columns: {audit['total_columns']}")
print(f"Date Range: {audit['date_range']['start']} to {audit['date_range']['end']}")
print(f"Latitude Span: {audit['geographic_coverage']['min_latitude']} to {audit['geographic_coverage']['max_latitude']}")
print(f"Longitude Span: {audit['geographic_coverage']['min_longitude']} to {audit['geographic_coverage']['max_longitude']}")
print(f"States Covered: {audit['geographic_coverage']['num_states']}")

## 3. Missing Value Analysis
Examine columns with significant missing values across all 7.728 million rows.

In [4]:
missing_df = pd.DataFrame.from_dict(audit['missing_summary'], orient='index')
missing_df = missing_df.sort_values(by='missing_percentage', ascending=False)
missing_df[missing_df['missing_percentage'] > 0.0]

## 4. Severity & Geographic Distribution
Analyze the outcome distribution and geographic concentration across states.

In [5]:
severity_df = pd.DataFrame(list(audit['severity_distribution'].items()), columns=['Severity', 'Count'])
severity_df['Percentage'] = (severity_df['Count'] / audit['total_rows']) * 100
print("=== Severity Distribution ===")
print(severity_df.to_string(index=False))

top_states_df = pd.DataFrame(audit['geographic_coverage']['top_states'], columns=['State', 'Incident_Count'])
top_states_df['Share(%)'] = (top_states_df['Incident_Count'] / audit['total_rows']) * 100
print("\n=== Top 10 States by Incident Volume ===")
print(top_states_df.head(10).to_string(index=False))

## 5. Road Context & Infrastructure Flags
Inspect the frequency of OpenStreetMap road infrastructure flags.

In [6]:
road_df = pd.DataFrame(list(audit['road_condition_flags'].items()), columns=['Road_Feature', 'True_Count'])
road_df['Share(%)'] = (road_df['True_Count'] / audit['total_rows']) * 100
road_df.sort_values(by='True_Count', ascending=False)

## 6. Numeric Weather Statistics & Sensor Outliers
Verify numeric weather ranges to identify sensor glitches and extreme outliers.

In [7]:
weather_df = pd.DataFrame.from_dict(audit['numeric_weather_summary'], orient='index')
weather_df

## 7. Strict Data Leakage Audit
Categorize every feature as Pre-Event vs Post-Event Leakage.

In [8]:
leakage_audit = pd.DataFrame([
    {"Feature": "End_Time", "Classification": "POST-EVENT LEAKAGE", "Reason": "Clearance completion timestamp; known only after emergency clearance."},
    {"Feature": "End_Lat", "Classification": "POST-EVENT LEAKAGE", "Reason": "Tail of traffic queue extent; measured post-incident."},
    {"Feature": "End_Lng", "Classification": "POST-EVENT LEAKAGE", "Reason": "Tail of traffic queue extent; measured post-incident."},
    {"Feature": "Distance(mi)", "Classification": "POST-EVENT LEAKAGE", "Reason": "Queue length of road impacted by the crash."},
    {"Feature": "Description", "Classification": "POST-EVENT LEAKAGE", "Reason": "Dispatcher/police accident report text describing collision."},
    {"Feature": "Severity", "Classification": "POST-EVENT RATING", "Reason": "Delay rating assigned post-accident; cannot predict accident occurrence."},
    {"Feature": "Weather_Timestamp", "Classification": "POTENTIAL LEAKAGE", "Reason": "Airport station observation timestamp; must not exceed prediction time T."}
])
leakage_audit

## 8. Summary & Technical Conclusions

1. **Suitable for Prototype?** Yes (Conditional Go) — 7.728M verified records over 7+ years with high-resolution coordinates and timestamps.
2. **Feasible Prediction Target**: 
   - **Primary**: Monitored Corridor Shift Risk (predicting elevated crash likelihood in active road cells over a 4-hour operational window).
   - **Alternative**: Conditional Severe Impact Risk ($Y = \mathbb{I}(\text{Severity} \ge 3)$).
3. **Missing Information**: True traffic exposure (AADT / vehicle volume) and continuous weather observations for non-crash periods are absent.
4. **Major Leakage Risks**: `End_Time`, `End_Lat`, `End_Lng`, `Distance(mi)`, `Description` are post-event and strictly excluded.
5. **Negative Sampling Problem**: As an incident-only dataset, negative samples must be restricted strictly to active monitored corridors during monitored periods, not generated randomly nationwide.
6. **Next Steps for Member 2**: Drop leakage columns, implement weather outlier clipping, construct corridor spatio-temporal tiles, and establish chronological train/val/test splits.